In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df = spark.table("silver_stock_prices")

# --- Window for rolling calculations ---
# Window functions are core PySpark — important to understand
window_7d = Window.partitionBy("ticker").orderBy("date").rowsBetween(-6, 0)
window_30d = Window.partitionBy("ticker").orderBy("date").rowsBetween(-29, 0)

df_gold = df \
    .withColumn("ma_7d", F.avg("close").over(window_7d)) \
    .withColumn("ma_30d", F.avg("close").over(window_30d)) \
    .withColumn("daily_return",
        (F.col("close") - F.lag("close", 1).over(
            Window.partitionBy("ticker").orderBy("date"))) 
        / F.lag("close", 1).over(
            Window.partitionBy("ticker").orderBy("date"))) \
    .withColumn("volatility_7d", F.stddev("close").over(window_7d)) \
    .withColumn("gold_created_at", F.current_timestamp())

df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_stock_features")

display(df_gold.orderBy("date", ascending=False).limit(10))